# Analisis Komparatif Algoritma Klasifikasi Sentimen dalam Mengukur Kesesuaian Star Rating dan Ulasan Pengguna Menggunakan Visualisasi Analitik
## Eksperimen: Multinomial NB, Linear SVM, & Logistic Regression
### Dataset: Google Play Store Reviews (12,495 Rows)

Notebook ini dirancang untuk memberikan analisis komprehensif mulai dari pemahaman data (EDA) hingga evaluasi model mendalam menggunakan dataset skala menengah yang kredibel secara akademik.

### 1. Inisialisasi dan Pemuatan Data

In [ ]:
import pandas as pd
import numpy as np
import re
import kagglehub
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, 
    balanced_accuracy_score, f1_score
)
from imblearn.over_sampling import RandomOverSampler

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

sns.set_theme(style="whitegrid", palette="pastel")
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
# Download dataset baru (12k+ rows)
path = kagglehub.dataset_download("prakharrathi25/google-play-store-reviews")
df_raw = pd.read_csv(f"{path}/reviews.csv")

# Preprocessing kolom agar sesuai dengan pipeline sebelumnya
df = df_raw[['content', 'score']].copy()
df = df.dropna(subset=['content']).rename(columns={"content": "review", "score": "star_rating"})

# Mapping Sentiment
df['sentiment'] = df['star_rating'].apply(lambda x:
    'positive' if x >= 4 else 'neutral' if x == 3 else 'negative'
)

df['review_len'] = df['review'].apply(lambda x: len(str(x)))
print(f"Dataset Loaded: {len(df)} rows")

### 2. Exploratory Data Analysis (EDA)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# 1. Distribusi Star Rating
sns.countplot(x='star_rating', data=df, ax=ax[0], palette='magma')
ax[0].set_title('Distribusi Star Rating (1-5)')

# 2. Distribusi Sentimen
df['sentiment'].value_counts().plot.pie(autopct='%1.1f%%', ax=ax[1], explode=[0.05]*3, shadow=True)
ax[1].set_title('Proporsi Sentimen Asli (Berdasarkan Rating)')
plt.show()

### 3. Text Preprocessing & TF-IDF

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = re.sub(r'[^a-z\s]', '', str(text).lower())
    return " ".join([lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words])

df['clean_review'] = df['review'].apply(clean_text)

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['clean_review'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Balancing Data
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(dual=False)
}

results = {}
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    preds = model.predict(X_test)
    results[name] = preds
    print(f"--- {name} Results ---")
    print(classification_report(y_test, preds))


### 4. Analisis Kesesuaian (Mismatch Analysis) & Tableau Export

In [ ]:
# Memilih model terbaik (misal Logistic Regression)
best_model_name = "Logistic Regression"
y_pred = results[best_model_name]

mismatch_df = df.iloc[y_test.index].copy()
mismatch_df['predicted_sentiment'] = y_pred
mismatch_df['is_correct'] = mismatch_df['sentiment'] == mismatch_df['predicted_sentiment']

# Simpan untuk Tableau
mismatch_df.to_csv("tableau_playstore_analysis.csv", index=False)
print("Data 12k ulasan berhasil diekspor untuk Tableau!")